In [1]:
import xarray as xr
import shutil

from pathlib import Path
import os
from os.path import join
import sys
import traceback
import glob
import numpy as np
import datetime

from joblib import Parallel, delayed
import joblib

from glob import glob

from functools import partial
import dask.array as da

import pandas as pd

from dateutil.parser import parse
from multiprocessing import Pool

from credit.pbs import get_num_cpus

import pickle

This notebook filters out invalid timestamps (too many NaN, incomplete channel list) and
- checks for duplicate timestamps
- then sorts all the timestamps and writes them to a valid_times file

In [ ]:
save_dir = "/glade/derecho/scratch/dkimpara/goes-cloud-dataset"
num_cpus = get_num_cpus()
num_cpus = 2

dir = "/glade/derecho/scratch/dkimpara/goes-cloud-dataset/2025"

channels = [4, 7, 8, 9, 10, 13]
outfile = r"/glade/derecho/scratch/dkimpara/goes-cloud-dataset/intermediate_files/file_valid_time_dict_2025_final.pkl"
valid_times_outfile = ""

ref_file_list = sorted([f for f in Path(dir).iterdir() if f"C{channels[0]:02}" in f.name])
# ref_file_list = sorted([
#     "/glade/derecho/scratch/dkimpara/goes-cloud-dataset/2025/2025-09-08_07Z_C04.nc",
#     "/glade/derecho/scratch/dkimpara/goes-cloud-dataset/2025/2025-09-18_15Z_C04.nc",
#     "/glade/derecho/scratch/dkimpara/goes-cloud-dataset/2025/2025-10-20_16Z_C04.nc",
# ])
ref_file_list = [Path(p) for p in ref_file_list]
chunked = np.array_split(ref_file_list, num_cpus - 1)


In [19]:
def get_valid_times_in_files(chunk):
    #dict of file, time pairs
    file_valid_time_dict = {}

    for file in chunk:
        try:
            time_str = file.name[:-7]
            paths = [join(file.parent ,f"{time_str}_C{channel:02}.nc") for channel in channels]

            existing = [p for p in paths if Path(p).exists()]

            if len(existing) != len(paths):
                for p in existing:
                    os.remove(p)
                    continue #skip this time because not all channels present
            
            drop_indices = set()

            datasets = [xr.open_dataset(p).sortby("t") for p in paths]
            def check_within_hour(ts):
                ts = pd.Timestamp(ts)
                return hour.hour == ts.hour
                
            file_valid_time_dict[str(file)] = []
            for t in datasets[0].t.values:
                if np.any([np.abs(t - ds.t.sel(t=t, method="nearest")) > pd.Timedelta(1, "m") for ds in datasets]):
                    continue

                hour = pd.Timestamp(parse(time_str[:-4] + "T" + time_str[-3:-1]))

                if np.any( [not check_within_hour(ds.t.sel(t=t, method="nearest").values) for ds in datasets ]):
                    continue

                for ds in datasets:
                    sub_ds = ds.sel(t=t, method="nearest")
                    if any((sub_ds.BT_or_R.isnull()).mean(dim=["lat", "lon"]) >= 0.25):
                        continue

                file_valid_time_dict[str(file)].append(t)
        except:
            print(f"error while processing {str(file)}")
            print(traceback.format_exc()) # This line is for getting traceback.
            print(sys.exc_info()[2]) # This line is getting for the error type.
            pass

    return file_valid_time_dict


In [ ]:
with Pool(num_cpus - 1) as p:
    result = p.map(get_valid_times_in_files, chunked)
    p.close()
    p.join()

res_dict = {}
for d in result:
    res_dict = res_dict | d

{'/glade/derecho/scratch/dkimpara/goes-cloud-dataset/2025/2025-09-08_07Z_C04.nc': [np.datetime64('2025-09-08T07:05:06.461218048'),
  np.datetime64('2025-09-08T07:15:06.467169024'),
  np.datetime64('2025-09-08T07:35:06.465833984'),
  np.datetime64('2025-09-08T07:45:06.467779968'),
  np.datetime64('2025-09-08T07:55:06.467828992')],
 '/glade/derecho/scratch/dkimpara/goes-cloud-dataset/2025/2025-09-18_15Z_C04.nc': [np.datetime64('2025-09-18T15:05:06.029361024'),
  np.datetime64('2025-09-18T15:15:06.031592064'),
  np.datetime64('2025-09-18T15:35:06.030300032'),
  np.datetime64('2025-09-18T15:45:06.038958976'),
  np.datetime64('2025-09-18T15:55:06.034168960')],
 '/glade/derecho/scratch/dkimpara/goes-cloud-dataset/2025/2025-10-20_16Z_C04.nc': [np.datetime64('2025-10-20T16:05:06.543997952'),
  np.datetime64('2025-10-20T16:25:06.538753024'),
  np.datetime64('2025-10-20T16:35:06.546004992'),
  np.datetime64('2025-10-20T16:45:06.315372032'),
  np.datetime64('2025-10-20T16:55:06.318770944')]}

In [ ]:
with open(outfile, "wb") as handle:
    pickle.dump(res_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
# check for duplicates
keys = sorted(list(res_dict.keys()))

def check_duplicates(i_key):
    i, key = i_key
    t_stamps = res_dict[key]
    for key_to_check in keys[i + 1:]:
        to_check = res_dict[key_to_check]
        # check for duplicates
        for ts in t_stamps:
            if ts in to_check:
                print(f"duplicate time found {ts} in {key} and {key_to_check}")

with Pool(num_cpus - 1) as p:
    result = p.map(check_duplicates, enumerate(keys[:-1]))
    p.close()
    p.join()


In [ ]:
times = np.concatenate(list(res_dict.values()))
times_sorted = np.sort(times)

with open(valid_times_outfile, 'wb') as f:
    np.save(f, times_sorted)

